# Phase 7 Worksheet — Image Retrieval (OCR + CLIP via Hugging Face)
`pip install pytesseract pillow transformers torch --break-system-packages` (plus the Tesseract binary itself: `brew install tesseract` on Mac). CLIP downloads from Hugging Face on first use, per your go-ahead.

**Corrected in this version:** the in-house Jina text embedding now goes through `embedder.embed_query()`. CLIP itself is unchanged — it's intentionally separate from the in-house wrapper. The vision-model section now also shows `ask_vision()` as the correct way to call `MODEL_QWEN2_5_VL_7B`, fixing the bug where `multimodal_chat()` had no real client for that model and sent the image in an invalid format.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../../wrapper_fix"))  # folder containing the corrected inhouse_wrappers.py
sys.path.append(os.path.abspath("../../inhouse_rag_capstone"))  # folder containing your real inhouse_llm.py

from inhouse_llm import MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL, MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B
from inhouse_wrappers import get_chat_model, InHouseEmbeddings, build_vision_messages, llm_for
from langchain_core.messages import SystemMessage, HumanMessage

embedder = InHouseEmbeddings()

def ask(system_prompt, user_prompt, model=MODEL_QWEN3_14B, max_tokens=200):
    """Drop-in replacement for the old multimodal_chat() text-only calls --
    correctly routed per-model via get_chat_model(), unlike inhouse_llm.py's
    own chat()/multimodal_chat() which always hit the Qwen3-14B endpoint."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]).content

def ask_vision(system_prompt, user_prompt, image_base64, model=MODEL_QWEN2_5_VL_7B, max_tokens=200):
    """Drop-in replacement for multimodal_chat() WITH an image -- uses the
    corrected image_url content-block format, and an actual client for the
    vision model (inhouse_llm.py never created one)."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke(build_vision_messages(system_prompt, user_prompt, image_base64)).content

import chromadb
client = chromadb.HttpClient(host="localhost", port=8000)  # adjust to your Chroma server
print("Setup OK")

## 1. OCR

In [ ]:
import pytesseract
from PIL import Image

# img = Image.open("error_screenshot.png")
# text = pytesseract.image_to_string(img)
# print(text)
print("Uncomment above and point at a real screenshot/diagram image.")

## 2. Qwen2.5-VL via the CORRECTED vision call
This is the fix for the original bug: `multimodal_chat()` had no client bound to the VL model's endpoint and sent the image as a raw base64 string in a plain text message. `ask_vision()` uses `get_chat_model(model=MODEL_QWEN2_5_VL_7B)` (the right endpoint) and `build_vision_messages()` (the right `image_url` content-block format).

In [ ]:
import base64

# with open("architecture_diagram.png", "rb") as f:
#     img_b64 = base64.b64encode(f.read()).decode()
#
# description = ask_vision(
#     "Describe what this diagram shows in 2 sentences.",
#     "What does this diagram show?",
#     img_b64,
# )
# print(description)
print("Uncomment above and point at a real image file to try the corrected vision call.")

## 3. CLIP image + text embeddings (shared space, unchanged from before)

In [ ]:
from transformers import CLIPModel, CLIPProcessor
import torch

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def clip_embed_text(text):
    inputs = clip_processor(text=[text], return_tensors="pt", padding=True)
    with torch.no_grad():
        return clip_model.get_text_features(**inputs)[0].numpy()

def clip_embed_image(image_path):
    img = Image.open(image_path)
    inputs = clip_processor(images=img, return_tensors="pt")
    with torch.no_grad():
        return clip_model.get_image_features(**inputs)[0].numpy()

text_vec = clip_embed_text("a diagram of a payment processing flow")
print("CLIP text embedding dim:", text_vec.shape)
# image_vec = clip_embed_image("architecture_diagram.png")
# print("CLIP image embedding dim:", image_vec.shape)

## 4. Reproducing this phase's teaser bug: mixing CLIP images with Jina text in one collection

In [ ]:
bad_collection = client.get_or_create_collection("phase7_mixed_bug")

# Simulating with text-only since we may not have a real image file here:
fake_clip_image_vec = clip_embed_text("architecture diagram").tolist()  # stand-in for a real image embedding
jina_text_vec = embedder.embed_query("This diagram shows the payment service architecture.")

bad_collection.upsert(
    ids=["clip_img", "jina_text"],
    embeddings=[fake_clip_image_vec[:len(jina_text_vec)] if len(fake_clip_image_vec) != len(jina_text_vec) else fake_clip_image_vec, jina_text_vec],
    documents=["[image]", "This diagram shows the payment service architecture."],
)
print("NOTE: if dims don't match, Chroma will error on add -- that mismatch itself")
print("is a visible symptom of the same root bug: incompatible embedding spaces.")

## 5. The fix: separate collections, linked by metadata

In [ ]:
coll_clip = client.get_or_create_collection("phase7_images_clip")
coll_text = client.get_or_create_collection("phase7_docs_jina")

doc_id = "doc_42"
coll_text.upsert(ids=["chunk_1"],
                  embeddings=[embedder.embed_query("See the architecture diagram below.")],
                  documents=["See the architecture diagram below."],
                  metadatas=[{"doc_id": doc_id, "section": "Architecture"}])

coll_clip.upsert(ids=["img_1"],
                  embeddings=[clip_embed_text("payment architecture diagram").tolist()],
                  documents=["[architecture_diagram.png]"],
                  metadatas=[{"doc_id": doc_id, "section": "Architecture"}])

text_hit = coll_text.query(query_embeddings=[embedder.embed_query("architecture diagram")], n_results=1)
linked_doc_id = text_hit["metadatas"][0][0]["doc_id"]
linked_section = text_hit["metadatas"][0][0]["section"]
linked_image = coll_clip.get(where={"$and": [{"doc_id": linked_doc_id}, {"section": linked_section}]})
print("Text hit:", text_hit["documents"][0])
print("Linked image via metadata:", linked_image["documents"])

## Teaser exercise
Query the CLIP collection with a CLIP TEXT embedding of 'a payment flow diagram' instead of going through the Jina collection at all — confirm CLIP's shared text-image space lets you search images directly. Then try `ask_vision()` on a real screenshot and confirm it's now actually reaching the VL model (check for no connection/404 errors, which is what the old broken code would likely have hit).